# 03 - Quick model checks

Fast, fold-local checks before running the full pipeline in `scripts/`. Every transformation is fitted on the training partition only.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd

from config import DATA_PROCESSED, RESULTS_DIR, FIGURES_DIR, SEED

df = pd.read_csv(DATA_PROCESSED / "cleaned_daily.csv", parse_dates=["timestamp"])
print(f"{len(df):,} station-days | {df.station_id.nunique()} stations | "
      f"{df.timestamp.min():%Y-%m-%d} to {df.timestamp.max():%Y-%m-%d}")

40,983 station-days | 95 stations | 2016-11-25 to 2026-04-06


## Temporal folds

Five expanding folds over whole dates with a one-day embargo, loaded from `folds/folds_temporal.json`.

In [2]:
import json
from config import FOLDS_DIR

folds = json.loads((FOLDS_DIR / 'folds_temporal.json').read_text(encoding='utf-8'))
print(folds['scheme'], '|', folds['n_splits'], 'folds | development stops before', folds['development_cutoff_exclusive'])
pd.DataFrame([
    {'fold': f['fold'], 'train': f['train_n_samples'], 'validation': f['validation_n_samples'],
     'validation_start': f['validation_start'][:10], 'validation_end': f['validation_end'][:10],
     'embargo_days': f['embargo_days']}
    for f in folds['folds']
])

temporal_panel_expanding | 5 folds | development stops before 2024-12-08T00:00:00Z


,fold,train,validation,validation_start,validation_end,embargo_days
0,1,3474,3392,2019-04-30,2020-10-16,1
1,2,6866,4072,2020-10-17,2021-10-24,1
2,3,10947,6033,2021-10-25,2022-11-18,1
3,4,16981,6001,2022-11-19,2024-01-30,1
4,5,22981,7663,2024-01-31,2024-12-07,1


## Naive baselines on fold 1

Persistence and seasonal-naive scores on the first outer fold.

In [3]:
from pm25_project.baselines import evaluate_baseline_fold

fold = folds['folds'][0]
train = df.iloc[fold['train_indices']]
valid = df.iloc[fold['validation_indices']]
scores, predictions = evaluate_baseline_fold(train, valid, 7)
pd.DataFrame(scores).round(3)

,model_id,rmse,mae,r2,n_samples
0,persistence,7.536,2.449,0.289,2192
1,seasonal_naive,11.355,4.325,-0.565,1986


## A linear benchmark with origin-safe features

Ridge regression on lag-1, rolling means and cyclic calendar terms.

In [4]:
from sklearn.linear_model import Ridge
from config import load_project_config
from pm25_project.features import build_origin_features
from pm25_project.metrics import regression_metrics

cfg, _ = load_project_config()
featured = build_origin_features(df, cfg)
featured = featured.assign(label=featured.groupby('station_id').pm25.shift(-1))
columns = ['pm25_lag1', 'pm25_roll3_mean', 'pm25_roll7_mean', 'day_of_week_sin', 'day_of_week_cos']
train = featured.loc[fold['train_indices']].dropna(subset=columns + ['label'])
valid = featured.loc[fold['validation_indices']].dropna(subset=columns + ['label'])
model = Ridge(alpha=1.0).fit(train[columns], train.label)
regression_metrics(valid.label.to_numpy(), model.predict(valid[columns]))

{'rmse': 4.118241537389173,
 'mae': 2.6345602150127627,
 'r2': 0.42225686178565236,
 'n_samples': 425}

These numbers are a sanity check on the plumbing, not the reported estimate. The reported development metrics come from `scripts/run_evaluation.py`; the CNN-LSTM is trained in `scripts/run_training.py`.